# Experimento 02 — Quanto do modelo depende de features-proxy?

> **Status:** *Sandbox / aquecimento.* Dataset atual não é o oficial da Ford. Objetivo é medir **sensibilidade do modelo** à presença de features sintéticas.

## Contexto — por que esse teste

No Experimento 01, XGBoost atingiu **86,3% de balanced accuracy** prevendo `perfil_latente` só com features pré-compra. Mas duas features — `organizacao_proxy` e `sensibilidade_preco_inicial` — juntas carregavam **34,7% da importância agregada**.

Essas features são **proxies de traços latentes** — inferências sobre o perfil do cliente, não medições diretas. No dataset real da Ford, elas:

- Podem existir com mesmo nome e significado (confirmação do Prof. Carlos de que será similar).
- Podem existir em forma diferente, exigindo derivação a partir de variáveis brutas (tempo entre cotação e fechamento, número de visitas, etc).
- Podem simplesmente não estar — caso extremo pessimista.

## Pergunta

**Qual é a faixa honesta de performance?** Rodamos 3 variantes:

| # | Cenário | O que remove |
|---|---|---|
| A | Otimista (base) | Nada — o 86,3% do Exp 01 |
| B | Realista | Remove os 3 traços latentes (`organizacao_proxy`, `sensibilidade_preco_inicial`, `tempo_decisao_dias`) |
| C | Pessimista | B + remove os add-ons de decisão (`plano_manutencao`, `garantia_estendida`, `aceitou_marketing`) |

O cenário C simula um dataset real só com demografia, uso e transação — o mínimo que qualquer CRM automotivo teria.

## 1. Setup e carga

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#6A994E", "#6C757D"]
sns.set_palette(PALETTE)

PROFILE_ORDER = ["fiel", "economico", "esquecido", "abandono"]
RANDOM_STATE = 42

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")


df = pd.read_parquet(DATA_PROCESSED / "ford_clientes_clean.parquet")
feature_dict = json.loads((DATA_PROCESSED / "feature_dictionary.json").read_text(encoding="utf-8"))
PRE_PURCHASE = feature_dict["pre_purchase"]
TARGET = "perfil_latente"

LABEL_MAP = {n: i for i, n in enumerate(PROFILE_ORDER)}
y = df[TARGET].map(LABEL_MAP).astype(int)
print(f"Rows: {len(df):,}  |  Pre-purchase features: {len(PRE_PURCHASE)}")

## 2. Definição dos 3 cenários

Cada cenário é uma *lista de colunas removidas*. O resto do pipeline é idêntico — só o que o modelo vê muda.

In [ ]:
LATENT_PROXIES = [
    "organizacao_proxy",
    "sensibilidade_preco_inicial",
    "tempo_decisao_dias",
]
DECISION_ADDONS = [
    "plano_manutencao",
    "garantia_estendida",
    "aceitou_marketing",
]

scenarios: dict[str, list[str]] = {
    "A_optimistic": [],
    "B_realistic": LATENT_PROXIES,
    "C_pessimistic": LATENT_PROXIES + DECISION_ADDONS,
}

for name, removed in scenarios.items():
    remaining = [c for c in PRE_PURCHASE if c not in removed]
    print(f"{name:<15}  removes {len(removed):>2}  keeps {len(remaining):>2}  (removed={removed})")

## 3. Loop de treino — mesmo modelo em cada cenário

In [ ]:
def build_pipeline(features: list[str]) -> Pipeline:
    """Builds preprocessor + XGBoost for a given feature subset.
    Monta preprocessador + XGBoost para um subconjunto de features."""
    numeric = [c for c in features if df[c].dtype != "object"]
    categorical = [c for c in features if df[c].dtype == "object"]
    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])
    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=len(PROFILE_ORDER),
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("pre", preprocessor), ("clf", clf)])


results: list[dict] = []
per_class: dict[str, dict] = {}

for scen_name, removed in scenarios.items():
    features = [c for c in PRE_PURCHASE if c not in removed]
    X = df[features]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    pipe = build_pipeline(features)
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)

    bal_acc = balanced_accuracy_score(y_te, pred)
    macro_f1 = f1_score(y_te, pred, average="macro")
    results.append({
        "scenario": scen_name,
        "n_features": len(features),
        "balanced_accuracy": round(bal_acc, 4),
        "macro_f1": round(macro_f1, 4),
        "removed": removed,
    })
    per_class[scen_name] = {
        "recall": f1_score(y_te, pred, average=None, labels=[LABEL_MAP[p] for p in PROFILE_ORDER]).tolist(),
    }
    print(f"{scen_name:<15}  features={len(features):>2}  bal_acc={bal_acc:.4f}  macro_f1={macro_f1:.4f}")

results_df = pd.DataFrame(results)
results_df

## 4. Visualização — a queda de performance por cenário

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall performance
# Performance agregada
scen_labels = ["A · otimista", "B · realista", "C · pessimista"]
colors = [PALETTE[3], PALETTE[0], PALETTE[1]]
axes[0].bar(scen_labels, results_df["balanced_accuracy"] * 100, color=colors)
for i, v in enumerate(results_df["balanced_accuracy"] * 100):
    axes[0].text(i, v, f"{v:.1f}%", ha="center", va="bottom")
axes[0].set_ylabel("Balanced Accuracy (%)")
axes[0].set_title("Performance agregada por cenário")
axes[0].set_ylim(0, 100)

# Per-class F1 (the detail that actually matters)
# F1 por classe (o detalhe que realmente importa)
f1_matrix = pd.DataFrame(
    {s: per_class[s]["recall"] for s in scenarios},
    index=PROFILE_ORDER,
) * 100
f1_matrix.columns = scen_labels
f1_matrix.plot(kind="bar", ax=axes[1], color=colors, width=0.75)
axes[1].set_ylabel("F1 por classe (%)")
axes[1].set_title("F1 por perfil em cada cenário")
axes[1].set_xticklabels(PROFILE_ORDER, rotation=0)
axes[1].legend(title="Cenário", loc="lower left", frameon=False)
axes[1].set_ylim(0, 100)

save_fig("robustness_comparison")
plt.show()

## 5. Diagnóstico — quais perfis mais sofrem?

A queda total é menos importante do que *qual classe desaba*. Se *esquecido* mantém F1 alto mesmo no cenário C, a tese estratégica do ForwardService continua viva.

In [ ]:
delta_AB = f1_matrix["B · realista"] - f1_matrix["A · otimista"]
delta_AC = f1_matrix["C · pessimista"] - f1_matrix["A · otimista"]

diagnosis = pd.DataFrame({
    "F1 A (otimista)": f1_matrix["A · otimista"].round(1),
    "F1 B (realista)": f1_matrix["B · realista"].round(1),
    "F1 C (pessimista)": f1_matrix["C · pessimista"].round(1),
    "Δ B-A": delta_AB.round(1),
    "Δ C-A": delta_AC.round(1),
})
diagnosis

## 6. Persistência dos aprendizados

Gravamos a faixa honesta em `learnings_experiment_02.json`. Quando o dataset oficial chegar, o script de classificação pode ler esse JSON pra saber qual score considerar "esperado" e qual alertar como suspeito.

In [ ]:
learnings = {
    "experiment": "02_robustness_without_proxies",
    "status": "sandbox — warm-up dataset, not Ford official",
    "scenarios": results,
    "honest_range": {
        "optimistic_pct": round(results_df["balanced_accuracy"].iloc[0] * 100, 1),
        "realistic_pct": round(results_df["balanced_accuracy"].iloc[1] * 100, 1),
        "pessimistic_pct": round(results_df["balanced_accuracy"].iloc[2] * 100, 1),
    },
    "per_class_f1": {
        s: {p: round(v, 4) for p, v in zip(PROFILE_ORDER, per_class[s]["recall"])}
        for s in scenarios
    },
    "decision_guidance": {
        "trustworthy_ceiling": "Use realistic (B) as the number to report externally.",
        "red_flag_if_above": round(results_df["balanced_accuracy"].iloc[0] * 100 + 2, 1),
        "alert_if_below": round(results_df["balanced_accuracy"].iloc[2] * 100 - 5, 1),
    },
}
out = DATA_PROCESSED / "learnings_experiment_02.json"
out.write_text(json.dumps(learnings, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {out}")
print()
print("Honest performance range (balanced accuracy):")
print(f"  Optimistic  (A): {learnings['honest_range']['optimistic_pct']}%")
print(f"  Realistic   (B): {learnings['honest_range']['realistic_pct']}%  <- report this")
print(f"  Pessimistic (C): {learnings['honest_range']['pessimistic_pct']}%")